# Catalog Exploration — Unity Catalog vs Legacy Hive Metastore
**Day 4 | Run this notebook on `dev-cluster` to understand exactly what you see in the Catalog UI**

---

## What you see in the Catalog tab (left sidebar)

```
Catalog tab
├── My organization                  ← your Unity Catalog Metastore (the invisible wrapper)
│     ├── dbw_ev_intelligence_dev    ← YOUR catalog (UC) — use this for everything
│     └── system                     ← Databricks internal catalog (read-only, ignore)
│
├── Shares received
│     └── samples                    ← Delta Sharing demo data (read-only, ignore)
│
└── Legacy
      └── hive_metastore             ← OLD workspace Hive metastore (pre-UC)
            └── default              ← schema inside old metastore (DO NOT use)
```

**The metastore itself** is not a clickable node — it is the account-level container  
that holds everything under "My organization". You see its catalogs, not the metastore object itself.

---

## What is `hive_metastore` (Legacy)?

When your Databricks workspace was first created, it auto-created a workspace-level  
Hive metastore. This is the old way — before Unity Catalog existed.  
When UC was enabled, Databricks moved it under **Legacy** instead of deleting it  
so existing tables would not break.

| | hive_metastore (Legacy) | dbw_ev_intelligence_dev (UC) |
|---|---|---|
| Namespace | `schema.table` (2-level) | `catalog.schema.table` (3-level) |
| Visible across workspaces | NO — this workspace only | YES — all workspaces in same region |
| Column-level security | No | Yes |
| Data lineage | No | Yes |
| External Locations / Volumes | No | Yes |
| Permissions tab in UI | No | Yes |

**Rule: Never create tables under `hive_metastore` for this project.**  
Always use `dbw_ev_intelligence_dev`.


## Cell 1 — See which catalog Databricks defaults to (without setting anything)

This is the risk — if you run SQL without specifying a catalog,  
Databricks may silently write to `hive_metastore` instead of your UC catalog.

In [ ]:
# Check the current default catalog — what would plain SQL use if you don't specify?
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
current_schema  = spark.sql("SELECT current_schema()").collect()[0][0]

print(f"Current catalog : {current_catalog}")
print(f"Current schema  : {current_schema}")
print()

if current_catalog == "hive_metastore":
    print("WARNING: Default is hive_metastore (Legacy). Run Cell 2 to switch to Unity Catalog.")
elif current_catalog == "dbw_ev_intelligence_dev":
    print("OK: Already on Unity Catalog.")
else:
    print(f"INFO: Current catalog is '{current_catalog}' — run Cell 2 to set explicitly.")

## Cell 2 — Always run this first: set catalog and schema to Unity Catalog

Run this at the top of every notebook before any SQL or table operations.  
It ensures all subsequent `SELECT`, `CREATE TABLE`, `INSERT` go to UC — not Legacy.

In [ ]:
UC_CATALOG = "dbw_ev_intelligence_dev"
UC_SCHEMA  = "default"

spark.sql(f"USE CATALOG {UC_CATALOG}")
spark.sql(f"USE SCHEMA {UC_SCHEMA}")

# Verify
current_catalog = spark.sql("SELECT current_catalog()").collect()[0][0]
current_schema  = spark.sql("SELECT current_schema()").collect()[0][0]

print(f"Active catalog : {current_catalog}")
print(f"Active schema  : {current_schema}")
print()
print("All SQL in this notebook now targets Unity Catalog.")
print(f"Short table names resolve to: {current_catalog}.{current_schema}.<table>")

## Cell 3 — List ALL catalogs visible to this workspace

Shows everything in the Catalog UI tree — UC catalogs, legacy, system, shares.

In [ ]:
print("=" * 55)
print("ALL CATALOGS VISIBLE TO THIS WORKSPACE")
print("=" * 55)

catalogs = spark.sql("SHOW CATALOGS").collect()

for row in catalogs:
    name = row[0]
    if name == "hive_metastore":
        label = "← Legacy Hive metastore (pre-UC) — DO NOT use for new work"
    elif name == "system":
        label = "← Databricks internal catalog (read-only)"
    elif name == "samples":
        label = "← Delta Sharing demo catalog (read-only)"
    elif name == "dbw_ev_intelligence_dev":
        label = "← YOUR Unity Catalog (use this for everything)"
    else:
        label = ""
    print(f"  {name:35s} {label}")

## Cell 4 — List schemas inside each catalog

Shows what's inside your UC catalog vs what's inside the Legacy hive_metastore.  
This is the second level of the namespace tree.

In [ ]:
print("=" * 55)
print("SCHEMAS INSIDE UNITY CATALOG: dbw_ev_intelligence_dev")
print("=" * 55)
uc_schemas = spark.sql("SHOW SCHEMAS IN dbw_ev_intelligence_dev").collect()
for row in uc_schemas:
    print(f"  dbw_ev_intelligence_dev.{row[0]}")

print()
print("=" * 55)
print("SCHEMAS INSIDE LEGACY: hive_metastore")
print("(shown for comparison — do not create tables here)")
print("=" * 55)
hive_schemas = spark.sql("SHOW SCHEMAS IN hive_metastore").collect()
for row in hive_schemas:
    print(f"  hive_metastore.{row[0]}")

## Cell 5 — List tables and volumes inside your UC schema

Shows the third level of the namespace — the actual objects you work with.

In [ ]:
print("=" * 55)
print("TABLES in dbw_ev_intelligence_dev.default")
print("=" * 55)
tables = spark.sql("SHOW TABLES IN dbw_ev_intelligence_dev.default").collect()
if tables:
    for row in tables:
        print(f"  dbw_ev_intelligence_dev.default.{row['tableName']}")
else:
    print("  (no tables yet — will be created in Silver layer)")

print()
print("=" * 55)
print("VOLUMES in dbw_ev_intelligence_dev.default")
print("=" * 55)
volumes = spark.sql("SHOW VOLUMES IN dbw_ev_intelligence_dev.default").collect()
if volumes:
    for row in volumes:
        print(f"  /Volumes/dbw_ev_intelligence_dev/default/{row['volume_name']}")
else:
    print("  (no volumes yet)")

## Cell 6 — Prove that `hive_metastore` is isolated to this workspace only

This cell shows the core difference between Legacy and Unity Catalog:  
- Tables in `hive_metastore` are only visible in THIS workspace  
- Tables in `dbw_ev_intelligence_dev` are visible to every workspace attached to the same metastore

In [ ]:
print("=" * 60)
print("SCOPE COMPARISON")
print("=" * 60)
print()
print("hive_metastore (Legacy):")
print("  Visible in     : THIS workspace only")
print("  Namespace      : schema.table (2-level)")
print("  Governance     : none (no column/row security, no lineage)")
print("  External Locs  : not supported")
print("  Volumes        : not supported")
print("  Status         : deprecated — Databricks is moving away from this")
print()
print("dbw_ev_intelligence_dev (Unity Catalog):")
print("  Visible in     : ALL workspaces attached to the same metastore")
print("  Namespace      : catalog.schema.table (3-level)")
print("  Governance     : column-level, row-level, lineage, permissions UI")
print("  External Locs  : yes — evdatalakedev-bronze/silver/gold")
print("  Volumes        : yes — /Volumes/dbw_ev_intelligence_dev/default/bronze-volume")
print("  Status         : current standard — use for all new work")
print()
print("=" * 60)
print("VERDICT: Always use dbw_ev_intelligence_dev for this project.")
print("=" * 60)

## Cell 7 — Browse the Bronze Volume path

Confirms the Volume is reachable and shows what's inside.  
This is the `/Volumes/...` path — no ADLS credentials needed in code.

In [ ]:
BRONZE_VOLUME = "/Volumes/dbw_ev_intelligence_dev/default/bronze-volume"

print(f"Listing root of Bronze Volume: {BRONZE_VOLUME}")
print()

try:
    items = dbutils.fs.ls(BRONZE_VOLUME)
    for item in items:
        kind = "DIR " if item.isDir() else "FILE"
        print(f"  {kind}  {item.path}")
    print()
    print("Volume is accessible — UC External Location + Storage Credential working correctly.")
except Exception as e:
    print(f"ERROR: {e}")
    print()
    print("Possible causes:")
    print("  1. Volume 'bronze-volume' does not exist yet — create it via Catalog UI")
    print("  2. Your user does not have READ VOLUME privilege")
    print("     Fix: GRANT READ VOLUME ON VOLUME dbw_ev_intelligence_dev.default.`bronze-volume` TO `your.email@company.com`;")
    print("  3. Cluster is not in Single User or Shared access mode (check Compute settings)")

## Cell 8 — Full namespace reference card

A quick-reference summary of every path format used in this project.

In [ ]:
print("=" * 65)
print("NAMESPACE REFERENCE CARD — EV Intelligence Project")
print("=" * 65)
print()
print("UNITY CATALOG (use these):")
print("  Metastore    : dbw_ev_intelligence_dev  (account-level, not in code)")
print("  Catalog      : dbw_ev_intelligence_dev")
print("  Schema       : default  (bronze/silver/gold schemas coming later)")
print("  Table (SQL)  : dbw_ev_intelligence_dev.default.<table_name>")
print("  Volume (code): /Volumes/dbw_ev_intelligence_dev/default/bronze-volume/")
print()
print("ADLS PATHS (used by ADF and in UC External Locations):")
print("  Bronze : abfss://bronze@evdatalakedev.dfs.core.windows.net/")
print("  Silver : abfss://silver@evdatalakedev.dfs.core.windows.net/")
print("  Gold   : abfss://gold@evdatalakedev.dfs.core.windows.net/")
print()
print("SOURCE BLOB (read-only, external):")
print("  wasbs://source@dataenggdailystorage.blob.core.windows.net/")
print("  realtime/charging_sessions/YYYY/MM/DD/HH/")
print()
print("LEGACY (do NOT use for new work):")
print("  hive_metastore.default.<table>  ← workspace-only, no governance")
print()
print("=" * 65)